# Pre-entrega: Persistencia y la integración de scripts ETL
**Curso:** Data Science II — Coderhouse  
**Módulo 9:** Automatización y Patrones de Datos  
**Estudiante:** Paulo Martin  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulo78marti-droid/Pre-entrega-8--Persistencia-y-la-integraci-n-de-scripts-ETL/blob/master/Pre-entrega_8_ETL.ipynb)

---

### Objetivo del Proyecto
Implementar un flujo automatizado de extracción, transformación y carga (**ETL**):
1. **Extracción (E):** Consumir la API pública del clima de Open-Meteo.
2. **Transformación (T):** Limpiar datos, tratar nulos, convertir tipos de datos y crear métricas con Pandas.
3. **Carga y Persistencia (L):** Almacenar los resultados en una base de datos SQLite con SQLAlchemy evitando duplicados.
4. **Consultas SQL:** Verificar los datos almacenados con queries analíticas.

## 1. Instalación e Importación de Librerías

In [ ]:
# Instalamos dependencias necesarias
!pip install requests pandas sqlalchemy python-dotenv --quiet

import os
import json
from pathlib import Path
from datetime import datetime
import requests
import pandas as pd
from sqlalchemy import create_engine, text, String, Integer, Float, Boolean, DateTime, Date
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

print("Librerías importadas correctamente.")

## 2. Configuración General
Definimos las coordenadas de la ciudad de Buenos Aires y las rutas de almacenamiento.

In [ ]:
CITY_NAME = "Buenos Aires"
LATITUDE = -34.61
LONGITUDE = -58.38
TIMEZONE = "America/Argentina/Buenos_Aires"
API_URL = "https://api.open-meteo.com/v1/forecast"

# Crear carpetas locales para persistencia y staging crudo
os.makedirs("data/raw", exist_ok=True)
DB_PATH = "data/weather_analytics.db"
DATABASE_URL = f"sqlite:///{DB_PATH}"

print(f"Configuración lista para {CITY_NAME}.")
print(f"Base de datos: {DATABASE_URL}")

## 3. Paso 1: Extracción (API pública Open-Meteo)
Realizamos la petición GET a la API del clima y guardamos una copia cruda de respaldo (*Staging Area*).

In [ ]:
def extract_weather_data():
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "timezone": TIMEZONE,
        "past_days": 7,
        "forecast_days": 3
    }
    
    print(f"[EXTRACT] Consultando API Open-Meteo para {CITY_NAME}...")
    response = requests.get(API_URL, params=params, timeout=15)
    response.raise_for_status()
    data = response.json()
    
    # Guardar copia cruda en data/raw/
    raw_file = Path("data/raw/weather_raw_latest.json")
    with open(raw_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        
    print(f"[EXTRACT] Datos crudos guardados en Staging: {raw_file.name}")
    return data

raw_data = extract_weather_data()

## 4. Paso 2: Transformación y Feature Engineering con Pandas
1. Convertimos el JSON a un DataFrame de Pandas.
2. Corregimos tipos de datos (`timestamp` a `datetime`).
3. Tratamos valores nulos (`dropna` para temperatura y fecha, `fillna` para lluvia y humedad).
4. Creamos columnas derivadas: `wind_category` y `is_rainy`.
5. Calculamos una tabla agregada con el resumen diario (`df_daily`).

In [ ]:
def clasificar_viento(velocidad):
    if velocidad < 15.0:
        return "Leve"
    elif velocidad < 30.0:
        return "Moderado"
    else:
        return "Fuerte"

def transform_weather_data(raw_data):
    hourly = raw_data["hourly"]
    
    # Crear DataFrame inicial
    df = pd.DataFrame({
        "timestamp": hourly["time"],
        "temperature_c": hourly["temperature_2m"],
        "humidity_pct": hourly["relative_humidity_2m"],
        "precipitation_mm": hourly["precipitation"],
        "wind_speed_kmh": hourly["wind_speed_10m"]
    })
    
    # Tipos de datos
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["city"] = CITY_NAME
    
    # Limpieza de nulos
    df = df.dropna(subset=["timestamp", "temperature_c"]).copy()
    df["precipitation_mm"] = df["precipitation_mm"].fillna(0.0)
    if df["humidity_pct"].isnull().any():
        df["humidity_pct"] = df["humidity_pct"].fillna(df["humidity_pct"].median())
        
    # Redondeos y Feature Engineering
    df["temperature_c"] = df["temperature_c"].round(2)
    df["precipitation_mm"] = df["precipitation_mm"].round(2)
    df["wind_speed_kmh"] = df["wind_speed_kmh"].round(2)
    df["wind_category"] = df["wind_speed_kmh"].apply(clasificar_viento)
    df["is_rainy"] = df["precipitation_mm"] > 0.0
    df["ingested_at"] = datetime.now()
    
    # Eliminar posibles duplicados
    df = df.drop_duplicates(subset=["city", "timestamp"]).copy()
    
    # Resumen diario agregado con groupby
    df["date"] = df["timestamp"].dt.date
    df_daily = df.groupby(["city", "date"]).agg(
        temp_min_c=("temperature_c", "min"),
        temp_max_c=("temperature_c", "max"),
        temp_avg_c=("temperature_c", "mean"),
        total_precipitation_mm=("precipitation_mm", "sum"),
        avg_wind_speed_kmh=("wind_speed_kmh", "mean"),
        rainy_hours_count=("is_rainy", "sum")
    ).reset_index()
    
    df_daily["temp_min_c"] = df_daily["temp_min_c"].round(2)
    df_daily["temp_max_c"] = df_daily["temp_max_c"].round(2)
    df_daily["temp_avg_c"] = df_daily["temp_avg_c"].round(2)
    df_daily["total_precipitation_mm"] = df_daily["total_precipitation_mm"].round(2)
    df_daily["avg_wind_speed_kmh"] = df_daily["avg_wind_speed_kmh"].round(2)
    df_daily["rainy_hours_count"] = df_daily["rainy_hours_count"].astype(int)
    df_daily["ingested_at"] = datetime.now()
    
    return df, df_daily

df_hourly, df_daily = transform_weather_data(raw_data)
print(f"[TRANSFORM] Filas horarias: {len(df_hourly)} | Días resumidos: {len(df_daily)}")
display(df_hourly.head(3))
display(df_daily.head(3))

## 5. Paso 3: Carga y Persistencia en SQLite con SQLAlchemy 2.0
Definimos los modelos con SQLAlchemy y persistimos los datos asegurando que no se creen duplicados si el proceso se vuelve a ejecutar.

In [ ]:
class Base(DeclarativeBase):
    pass

class WeatherHourly(Base):
    __tablename__ = "weather_hourly"
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    city: Mapped[str] = mapped_column(String(50), nullable=False)
    timestamp: Mapped[datetime] = mapped_column(DateTime, nullable=False)
    temperature_c: Mapped[float] = mapped_column(Float, nullable=False)
    humidity_pct: Mapped[float] = mapped_column(Float, nullable=True)
    precipitation_mm: Mapped[float] = mapped_column(Float, default=0.0)
    wind_speed_kmh: Mapped[float] = mapped_column(Float, nullable=False)
    wind_category: Mapped[str] = mapped_column(String(30), nullable=False)
    is_rainy: Mapped[bool] = mapped_column(Boolean, nullable=False)
    ingested_at: Mapped[datetime] = mapped_column(DateTime, default=datetime.now)

class WeatherDailySummary(Base):
    __tablename__ = "weather_daily_summary"
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    city: Mapped[str] = mapped_column(String(50), nullable=False)
    date: Mapped[datetime] = mapped_column(Date, nullable=False)
    temp_min_c: Mapped[float] = mapped_column(Float, nullable=False)
    temp_max_c: Mapped[float] = mapped_column(Float, nullable=False)
    temp_avg_c: Mapped[float] = mapped_column(Float, nullable=False)
    total_precipitation_mm: Mapped[float] = mapped_column(Float, default=0.0)
    avg_wind_speed_kmh: Mapped[float] = mapped_column(Float, nullable=False)
    rainy_hours_count: Mapped[int] = mapped_column(Integer, default=0)
    ingested_at: Mapped[datetime] = mapped_column(DateTime, default=datetime.now)

engine = create_engine(DATABASE_URL)
Base.metadata.create_all(engine)
print("Tablas SQLite creadas y verificadas.")

# Inserción evitando duplicados
def load_without_duplicates(engine, df_hourly, df_daily):
    with engine.connect() as conn:
        res_h = conn.execute(text("SELECT timestamp FROM weather_hourly WHERE city = :c"), {"c": CITY_NAME}).fetchall()
        ts_existentes = set(pd.to_datetime(f[0]) for f in res_h)
        
        res_d = conn.execute(text("SELECT date FROM weather_daily_summary WHERE city = :c"), {"c": CITY_NAME}).fetchall()
        fechas_existentes = set(str(f[0])[:10] for f in res_d)
        
    # Filtrar solo registros nuevos
    nuevos_h = df_hourly[~pd.to_datetime(df_hourly["timestamp"]).isin(ts_existentes)]
    nuevos_d = df_daily[~df_daily["date"].astype(str).isin(fechas_existentes)]
    
    if not nuevos_h.empty:
        nuevos_h.to_sql("weather_hourly", con=engine, if_exists="append", index=False)
        print(f"[LOAD] Se insertaron {len(nuevos_h)} registros horarios nuevos.")
    else:
        print("[LOAD] Registros horarios ya existentes: 0 duplicados.")
        
    if not nuevos_d.empty:
        nuevos_d.to_sql("weather_daily_summary", con=engine, if_exists="append", index=False)
        print(f"[LOAD] Se insertaron {len(nuevos_d)} resúmenes diarios nuevos.")
    else:
        print("[LOAD] Resúmenes diarios ya existentes: 0 duplicados.")

load_without_duplicates(engine, df_hourly, df_daily)

## 6. Paso 4: Verificación con Consultas SQL Analíticas
Ejecutamos 3 consultas analíticas en SQL sobre la base de datos para auditar los datos persistidos.

In [ ]:
print("=== CONSULTAS ANALÍTICAS SQL SOBRE SQLITE ===\n")

# 1. Horas más calurosas
df_calor = pd.read_sql("""
    SELECT timestamp, temperature_c, wind_speed_kmh, wind_category
    FROM weather_hourly
    ORDER BY temperature_c DESC
    LIMIT 5;
""", con=engine)
print("1. Top 5 horas con mayor temperatura:")
display(df_calor)

# 2. Días con precipitaciones acumuladas
df_lluvias = pd.read_sql("""
    SELECT date, total_precipitation_mm, rainy_hours_count, temp_avg_c
    FROM weather_daily_summary
    WHERE total_precipitation_mm > 0
    ORDER BY total_precipitation_mm DESC;
""", con=engine)
print("\n2. Días con registro de lluvias:")
display(df_lluvias)

# 3. Distribución de horas por categoría de viento
df_viento = pd.read_sql("""
    SELECT wind_category, COUNT(*) AS horas, ROUND(AVG(temperature_c), 2) AS temp_promedio
    FROM weather_hourly
    GROUP BY wind_category
    ORDER BY horas DESC;
""", con=engine)
print("\n3. Horas por categoría de viento:")
display(df_viento)

## 7. Conclusiones y Cierre
- El pipeline ETL cumplió todas las etapas: **Extracción** desde API REST, **Transformación** y limpieza con Pandas, y **Carga** en SQLite con SQLAlchemy 2.0.
- La persistencia es **idempotente**, protegiendo la base de datos contra registros duplicados.
- Los datos quedaron estructurados y disponibles para análisis o entrenamiento de modelos de Machine Learning.